# 💎 Advanced Encryption Standard (AES / Rijndael) from Scratch
### Mata Kuliah: MA4151 - Kriptografi
**Berbasis Teks Karakter Asli, Matriks Desimal (0-255), dan Bit Biner (0/1)**

---

## 🎯 Tujuan Pembelajaran
1. Memahami bahwa **128 bit = 16 karakter teks** (misal kata `"KRIPTOGRAFI ITB!"`).
2. Menyusun 16 karakter teks dalam **Matriks State $4 \times 4$** berurutan kolom demi kolom.
3. Mengimplementasikan operasi aljabar Lapangan Terhingga $GF(2^8)$ dan pembentukan Kotak-S secara matematis.
4. Menguasai 4 transformasi ronde AES dalam angka desimal biasa:
   - `SubBytes`
   - `ShiftRows`
   - `MixColumns`
   - `AddRoundKey`
5. Melakukan enkripsi teks nyata dan analisis difusi kilat.


---
## 1. Fondasi Aritmetika Lapangan Galois $GF(2^8)$


In [ ]:
# Aritmetika Lapangan Terhingga GF(2^8) (Desimal 0-255)

def xtime(a: int) -> int:
    hi_bit = a & 0x80
    shifted = (a << 1) & 0xFF
    return shifted ^ 0x1B if hi_bit else shifted

def gf_mult(a: int, b: int) -> int:
    """Perkalian di GF(2^8) (Russian Peasant Multiplication)."""
    res = 0
    for _ in range(8):
        if b & 1:
            res ^= a
        a = xtime(a)
        b >>= 1
    return res

def gf_inv(a: int) -> int:
    """Invers multiplikatif menggunakan a^(254) mod m(x)."""
    if a == 0:
        return 0
    res = 1
    base = a
    exp = 254
    while exp > 0:
        if exp & 1:
            res = gf_mult(res, base)
        base = gf_mult(base, base)
        exp >>= 1
    return res

# Pembangkitan S-Box Aljabar
S_BOX = [0] * 256
INV_S_BOX = [0] * 256
for i in range(256):
    inv = gf_inv(i)
    s = inv
    for shift in (1, 2, 3, 4):
        s ^= ((inv << shift) | (inv >> (8 - shift))) & 0xFF
    s ^= 0x63
    S_BOX[i] = s
    INV_S_BOX[s] = i

print(f"S-Box untuk desimal 0   : {S_BOX[0]}  (Biner: {S_BOX[0]:08b})")
print(f"S-Box untuk desimal 75  : {S_BOX[75]} (Biner: {S_BOX[75]:08b})")


---
## 2. Pembangkitan Matriks State $4 \times 4$ dari Teks 16 Karakter


In [ ]:
def text_to_state(text_16: str) -> list[list[int]]:
    """Mengonversi 16 karakter teks menjadi matriks State 4x4 desimal (column-major)."""
    assert len(text_16) == 16, "Teks harus tepat 16 karakter."
    decimals = list(text_16.encode('ascii'))
    state = [[0] * 4 for _ in range(4)]
    for col in range(4):
        for row in range(4):
            state[row][col] = decimals[col * 4 + row]
    return state

def state_to_text(state: list[list[int]]) -> str:
    """Mengonversi matriks State 4x4 kembali ke teks string."""
    decimals = []
    for col in range(4):
        for row in range(4):
            decimals.append(state[row][col])
    return bytes(decimals).decode('ascii', errors='replace')

def print_state_decimal(state: list[list[int]], label: str = "State:"):
    print(label)
    for r in range(4):
        print("  [" + " ".join(f"{state[r][c]:3d}" for c in range(4)) + "]")

pesan_aes = "KRIPTOGRAFI ITB!" # 16 karakter
state_awal = text_to_state(pesan_aes)
print_state_decimal(state_awal, f"Matriks State Desimal untuk '{pesan_aes}':")


---
## 3. Empat Transformasi Ronde AES


In [ ]:
def sub_bytes(state):
    for r in range(4):
        for c in range(4):
            state[r][c] = S_BOX[state[r][c]]

def inv_sub_bytes(state):
    for r in range(4):
        for c in range(4):
            state[r][c] = INV_S_BOX[state[r][c]]

def shift_rows(state):
    state[1] = state[1][1:] + state[1][:1] # Geser kiri 1
    state[2] = state[2][2:] + state[2][:2] # Geser kiri 2
    state[3] = state[3][3:] + state[3][:3] # Geser kiri 3

def inv_shift_rows(state):
    state[1] = state[1][3:] + state[1][:3]
    state[2] = state[2][2:] + state[2][:2]
    state[3] = state[3][1:] + state[3][:1]

def mix_columns(state):
    for c in range(4):
        s0, s1, s2, s3 = state[0][c], state[1][c], state[2][c], state[3][c]
        state[0][c] = gf_mult(0x02, s0) ^ gf_mult(0x03, s1) ^ s2 ^ s3
        state[1][c] = s0 ^ gf_mult(0x02, s1) ^ gf_mult(0x03, s2) ^ s3
        state[2][c] = s0 ^ s1 ^ gf_mult(0x02, s2) ^ gf_mult(0x03, s3)
        state[3][c] = gf_mult(0x03, s0) ^ s1 ^ s2 ^ gf_mult(0x02, s3)

def inv_mix_columns(state):
    for c in range(4):
        s0, s1, s2, s3 = state[0][c], state[1][c], state[2][c], state[3][c]
        state[0][c] = gf_mult(0x0E, s0) ^ gf_mult(0x0B, s1) ^ gf_mult(0x0D, s2) ^ gf_mult(0x09, s3)
        state[1][c] = gf_mult(0x09, s0) ^ gf_mult(0x0E, s1) ^ gf_mult(0x0B, s2) ^ gf_mult(0x0D, s3)
        state[2][c] = gf_mult(0x0D, s0) ^ gf_mult(0x09, s1) ^ gf_mult(0x0E, s2) ^ gf_mult(0x0B, s3)
        state[3][c] = gf_mult(0x0B, s0) ^ gf_mult(0x0D, s1) ^ gf_mult(0x09, s2) ^ gf_mult(0x0E, s3)

def add_round_key(state, round_key):
    for r in range(4):
        for c in range(4):
            state[r][c] ^= round_key[r][c]

print("Transformasi ronde AES siap!")


---
## 4. Key Expansion & Algoritma Enkripsi AES-128 Lengkap


In [ ]:
RCON = [0x00, 0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1B, 0x36]

def aes_key_expansion(key_text_16: str) -> list[list[list[int]]]:
    decimals = list(key_text_16.encode('ascii'))
    w = [decimals[4*i : 4*i+4] for i in range(4)]
    for i in range(4, 44):
        temp = list(w[i-1])
        if i % 4 == 0:
            temp = [S_BOX[b] for b in (temp[1:] + temp[:1])]
            temp[0] ^= RCON[i // 4]
        w.append([w[i-4][j] ^ temp[j] for j in range(4)])
    round_keys = []
    for r in range(11):
        rk = []
        for row in range(4):
            rk.append([w[r*4 + col][row] for col in range(4)])
        round_keys.append(rk)
    return round_keys

def aes_encrypt_block(state: list[list[int]], round_keys: list[list[list[int]]], verbose: bool = False):
    st = [row[:] for row in state]
    add_round_key(st, round_keys[0])
    for r in range(1, 10):
        sub_bytes(st)
        shift_rows(st)
        mix_columns(st)
        add_round_key(st, round_keys[r])
        if verbose and r == 1:
            print_state_decimal(st, "State Desimal setelah Ronde 1:")
    # Ronde 10 (Tanpa MixColumns)
    sub_bytes(st)
    shift_rows(st)
    add_round_key(st, round_keys[10])
    return st

def aes_decrypt_block(state: list[list[int]], round_keys: list[list[list[int]]]):
    st = [row[:] for row in state]
    add_round_key(st, round_keys[10])
    for r in range(9, 0, -1):
        inv_shift_rows(st)
        inv_sub_bytes(st)
        add_round_key(st, round_keys[r])
        inv_mix_columns(st)
    inv_shift_rows(st)
    inv_sub_bytes(st)
    add_round_key(st, round_keys[0])
    return st


---
## 5. Eksekusi Enkripsi Teks Nyata: `"KRIPTOGRAFI ITB!"` dengan Kunci `"KUNCI RAHASIAKU!"`


In [ ]:
pesan_aes = "KRIPTOGRAFI ITB!"
kunci_aes = "KUNCI RAHASIAKU!"

state_in = text_to_state(pesan_aes)
rkeys = aes_key_expansion(kunci_aes)

print("=== ENKRIPSI AES-128 ===")
print_state_decimal(state_in, f"State Teks Asli ('{pesan_aes}'):")
print_state_decimal(rkeys[0], f"Subkunci Ronde 0 K0 ('{kunci_aes}'):")

state_out = aes_encrypt_block(state_in, rkeys, verbose=True)

print_state_decimal(state_out, "\nState Ciphertext Akhir (Desimal):")

# Dekripsi
state_decrypted = aes_decrypt_block(state_out, rkeys)
teks_kembali = state_to_text(state_decrypted)
print(f"\nHasil Dekripsi Kembali: '{teks_kembali}' (100% Cocok!)")
